# 29 · Load: CPI-U, New York metro and U.S. → `staging`

**Source card:** `bls_cpi.md` (§6 issues, §7 staging table). Explored in: notebook 12.

| Table | One row = | Expected rows |
|---|---|---|
| `stg_bls__cpi` | CPI series × year × period (month, or `M13` annual average), 2015+ | 302 |

Two series: `CUURS12ASA0` (NY-Newark-Jersey City, all items, not seasonally adjusted) and `CUUR0000SA0` (U.S. city average, for comparison). The deflator to 2025 dollars is built in core from the annual averages. October 2025 is missing (federal shutdown; BLS collected no data): the file shows `-`, staged as NULL. BLS still published a 2025 annual average (NY metro 345.5), so the deflator has every year 2015–2025.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "29-load-cpi.ipynb"
SERIES = ["CUURS12ASA0", "CUUR0000SA0"]
conn = connect()

F = raw_files("bls_cu")
def rd(stem: str):
    row = F[F.file.str.contains(f"/{stem}_")].iloc[-1]
    d = pd.read_csv(row.path, sep="\t", dtype=str, keep_default_na=False)
    return d.rename(columns=str.strip).apply(lambda s: s.str.strip()), row.file

data, f_data = rd("cu-data-1-AllItems")
series, _ = rd("cu-series")
area, _ = rd("cu-area")

## 1. Filter, parse, label

In [2]:
c = data[data.series_id.isin(SERIES) & (data.year.astype(int) >= 2015)].copy()
log("keep NY metro + U.S. all-items series, 2015+", len(data), len(c))
assert not c.duplicated(["series_id", "year", "period"]).any()
assert c.period.str.fullmatch(r"M(0[1-9]|1[0-3])").all(), sorted(set(c.period))
v = pd.to_numeric(c.value, errors="coerce")
bad = c.value[v.isna()]
print("Non-numeric values -> NULL:", bad.value_counts().to_dict() or "none")
c["index_value"] = v.astype("float64")
c["year"] = c.year.astype(int).astype("Int64")
c["period_type"] = c.period.map(lambda p: "annual_avg" if p == "M13" else "month")
c["month"] = c.period.str[1:].astype(int).where(c.period != "M13").astype("Int64")
c["area_code"] = c.series_id.str[4:8]
c = c.merge(area[["area_code", "area_name"]], on="area_code", how="left")
assert c.area_name.notna().all()
c["footnote_code"] = c.footnote_codes.replace("", None)

grid = c[c.series_id == "CUURS12ASA0"].pivot_table(index="year", columns="period", values="index_value", aggfunc="first")
print("NY metro, periods missing by year:", {int(y): [p for p in [f"M{m:02d}" for m in range(1, 14)] if pd.isna(grid.loc[y].get(p))]
                                             for y in grid.index if grid.loc[y].isna().any() or len(grid.columns) < 13})
display(grid.loc[2019:].round(1))

  keep NY metro + U.S. all-items series, 2015+: 64,007 -> 302 rows (-63,705)
Non-numeric values -> NULL: {'-': 2}
NY metro, periods missing by year: {2025: ['M10'], 2026: ['M09', 'M10', 'M11', 'M12', 'M13']}


period,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11,M12,M13
year,,,,,,,,,,,,,
2019,275.1,275.8,276.6,277.4,278.1,278.8,278.8,279.4,279.3,279.3,279.5,279.8,278.2
2020,282.0,282.6,282.0,280.6,282.1,282.3,283.6,283.5,284.6,284.1,283.3,284.4,282.9
2021,285.5,286.5,287.5,289.5,291.0,293.9,293.6,293.9,295.5,296.5,297.5,296.9,292.3
2022,300.2,301.2,305.0,307.8,309.2,313.6,312.6,313.3,313.9,314.3,315.0,315.7,310.1
2023,318.2,319.3,319.0,319.2,320.0,321.3,322.5,324.4,325.6,325.3,324.5,324.7,322.0
2024,328.0,328.6,329.8,331.3,332.6,334.8,335.6,336.5,337.9,338.2,338.5,338.6,334.2
2025,341.1,342.3,342.5,344.0,343.9,346.3,346.4,347.3,348.1,NaN,348.6,350.0,345.5
2026,350.9,353.4,356.2,359.8,361.4,360.5,362.2,362.3,NaN,NaN,NaN,NaN,NaN


## 2. Tie-out: the annual average is the mean of the months (nb 12)

Checked for NY metro years with all 12 months published.

In [3]:
months = c[(c.series_id == "CUURS12ASA0") & (c.period_type == "month")].groupby("year").index_value.agg(["mean", "count"])
annual = c[(c.series_id == "CUURS12ASA0") & (c.period_type == "annual_avg")].set_index("year").index_value
t = months.join(annual.rename("published_annual"))
t = t[t["count"] == 12]
t["diff"] = (t["mean"] - t.published_annual).round(3)
display(t)
assert t["diff"].abs().max() <= 0.01

,mean,count,published_annual,diff
year,,,,
2015,260.558167,12,260.558,0.0
2016,263.364917,12,263.365,-0.0
2017,268.520417,12,268.520,0.0
2018,273.640500,12,273.641,-0.0
2019,278.164167,12,278.164,0.0
2020,282.919583,12,282.920,-0.0
2021,292.302583,12,292.303,-0.0
2022,310.141333,12,310.141,0.0
2023,321.997917,12,321.998,-0.0


## 3. Stage

In [4]:
stg = (c[["series_id", "area_code", "area_name", "year", "period", "period_type", "month", "index_value", "footnote_code"]]
       .assign(source_file=f_data).sort_values(["series_id", "year", "period"]).reset_index(drop=True))
display(stage(stg, "stg_bls__cpi", ["series_id", "year", "period"], expected_rows=302, source_key="bls_cu", notebook=NB, conn=conn,
              comment="BLS CPI-U all items, NY-Newark-Jersey City (S12A) and U.S. city average, NSA, 1982-84=100. M13 = annual average."))
if conn is not None:
    conn.close()

  stg_bls__cpi: 302 rows x 10 cols, key ['series_id', 'year', 'period'] unique | local data/interim/staging/stg_bls__cpi.parquet | DB: 302


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_bls__cpi,302,10,series_id + year + period,bls_cu,29-load-cpi.ipynb,302,2026-10-14T21:22:38+00:00
